In [1]:
from tensorflow.keras import layers,models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [2]:

img_height,img_width = 128,128
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 30,
    zoom_range = 0.3,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    horizontal_flip = True,
    brightness_range=[0.7,1.3],
    validation_split = 0.2,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)
base_model.trainable = False

In [3]:

train_datagen = train_datagen.flow_from_directory(
    'Dataset/train',
    target_size=(img_height,img_width),
    class_mode = "binary",
    batch_size = batch_size  
)

test_datagen = test_datagen.flow_from_directory(
    'Dataset/test',
    target_size=(img_height,img_width),
    class_mode = "binary",
    batch_size = batch_size  
)

val_datagen = val_datagen.flow_from_directory(
    'Dataset/val',
    target_size=(img_height,img_width),
    class_mode = "binary",
    batch_size = batch_size  
)



Found 2400 images belonging to 2 classes.
Found 460 images belonging to 2 classes.
Found 468 images belonging to 2 classes.


In [4]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    
)

In [5]:
model.fit(train_datagen,validation_data = val_datagen,epochs = 20)

Epoch 1/20
75/75 [==============================] - 59s 766ms/step - loss: 0.5268 - accuracy: 0.7375 - val_loss: 0.6739 - val_accuracy: 0.6752
Epoch 2/20
75/75 [==============================] - 56s 743ms/step - loss: 0.3721 - accuracy: 0.8392 - val_loss: 0.5347 - val_accuracy: 0.7479
Epoch 3/20
75/75 [==============================] - 54s 723ms/step - loss: 0.3313 - accuracy: 0.8596 - val_loss: 0.5282 - val_accuracy: 0.7607
Epoch 4/20
75/75 [==============================] - 54s 716ms/step - loss: 0.2959 - accuracy: 0.8729 - val_loss: 0.5673 - val_accuracy: 0.7671
Epoch 5/20
75/75 [==============================] - 54s 722ms/step - loss: 0.2888 - accuracy: 0.8808 - val_loss: 0.5319 - val_accuracy: 0.7756
Epoch 6/20
75/75 [==============================] - 54s 716ms/step - loss: 0.2698 - accuracy: 0.8892 - val_loss: 0.5527 - val_accuracy: 0.7671
Epoch 7/20
75/75 [==============================] - 54s 716ms/step - loss: 0.2588 - accuracy: 0.8946 - val_loss: 0.6913 - val_accuracy: 0.7500

In [6]:
test_loss, test_acc = model.evaluate(test_datagen)

print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

15/15 [==============================] - 6s 369ms/step - loss: 0.2745 - accuracy: 0.9043
Test Accuracy: 0.904347836971283
Test Loss: 0.27445143461227417


In [7]:
print(train_datagen.class_indices)
print(test_datagen.class_indices)
print(val_datagen.class_indices)

{'real_images': 0, 'screen_captured_images': 1}
{'real_images': 0, 'screen_captured_images': 1}
{'real_images': 0, 'screen_captured_images': 1}


In [8]:
model.save("screen_image_detector.keras")

In [1]:
import tensorflow as tf
import numpy as np
import cv2
model = tf.keras.models.load_model("screen_image_detector.keras")
IMG_SIZE = 128

def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    prediction = model.predict(img)[0][0]
    
    if prediction > 0.5:
        print("screen image")
    else:
        print("Normal image")

predict_image(r"C:\Users\User\Downloads\download.png")

1/1 [==============================] - 0s 500ms/step
screen image
